In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import geopandas as gpd
import contextily as ctx
import rasterio
from tqdm import tqdm

In [2]:
eq_scenario_id = "bssc2014ssanandreasnsbssbbgc_m7p64_se"
expansion_scenario = "conservative"

In [3]:
pypsa_data_dir = "/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/eq_handoff_fullcats"
agg_type = "disagg"
asset_fp = os.path.join(pypsa_data_dir, expansion_scenario, agg_type, "assets_all.csv")
asset_df = pd.read_csv(asset_fp)

asset_df['asset_id'] = range(len(asset_df))

buses_fp = os.path.join(pypsa_data_dir, expansion_scenario, agg_type, "buses_all.csv")
buses_df = pd.read_csv(buses_fp)

In [4]:
line_ind = asset_df[asset_df['bus']=='CATS_1875'].index[0]
site_lon, site_lat = asset_df['lon'][line_ind], asset_df['lat'][line_ind]
print(f"Site longitude: {site_lon}, Site latitude: {site_lat}")

Site longitude: -117.0698252, Site latitude: 33.68098774


In [5]:
eq_scenario_dir = f"/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/{eq_scenario_id}"
pga_raster_fp = os.path.join(eq_scenario_dir, "raster", "pga.fit")
# 1. Open the file
with rasterio.open(pga_raster_fp,
    crs="EPSG:4326") as src:
    # Query metadata
    print("Metadata Summary:")
    print(f"Dimensions: {src.width}x{src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    
    # 2. Query and read pixel data (e.g., Band 1)
    # raster_data = src.read(1)
    row, col = src.index(site_lon, site_lat)
    print(f"Pixel coordinates for the site: row={row}, col={col}")
    # 3. Read the pixel value for the site
    if 0 <= row < src.height and 0 <= col < src.width:
        site_pga = src.read(1)[row, col]
        print(f"PGA value for the site: {site_pga} % g")
    else:
        site_pga = 0.0
        print("Site is outside the raster bounds.")

Metadata Summary:
Dimensions: 810x613
Number of bands: 1
Coordinate Reference System (CRS): None
Pixel coordinates for the site: row=318, col=343
PGA value for the site: 16.113948822021484 % g


In [6]:
columns_pelicun_needs = [
    'lon', 'lat', 'asset_type', 'v_nom_kv', 'new_mw', 'asset_id'
]

asset_df_pelicun = asset_df[columns_pelicun_needs]

In [7]:
combined_assets = buses_df[['lon', 'lat', 'v_nom_kv', 'bus']].copy()
for col in columns_pelicun_needs:
    if col not in combined_assets.columns:
        combined_assets[col] = None
for col in combined_assets.columns:
    if col not in asset_df_pelicun.columns:
        asset_df_pelicun[col] = None
combined_assets['asset_type'] = 'Substation'
combined_assets = pd.concat([combined_assets, asset_df_pelicun], ignore_index=True)
combined_assets['id'] = range(len(combined_assets))
print(combined_assets.shape, buses_df.shape, asset_df_pelicun.shape)

(11930, 8) (8899, 7) (3031, 7)


In [8]:
combined_assets['site_pga'] = 0.0

In [9]:
with rasterio.open(pga_raster_fp,
    crs="EPSG:4326") as src:
    # Query metadata
    print("Metadata Summary:")
    print(f"Dimensions: {src.width}x{src.height}")
    print(f"Number of bands: {src.count}")
    print(f"Coordinate Reference System (CRS): {src.crs}")
    
    # 2. Query and read pixel data (e.g., Band 1)
    for site_id in tqdm(combined_assets.index, total=len(combined_assets)):
        site_lon, site_lat = combined_assets['lon'][site_id], combined_assets['lat'][site_id]
        row, col = src.index(site_lon, site_lat)
        # 3. Read the pixel value for the site
        if 0 <= row < src.height and 0 <= col < src.width:
            site_pga = src.read(1)[row, col]
            combined_assets.loc[site_id, 'site_pga'] = site_pga
            # print(f"PGA value for the site: {site_pga}")
        else:
            site_pga = 0.0
            combined_assets.loc[site_id, 'site_pga'] = site_pga
            # print("Site is outside the raster bounds.")

Metadata Summary:
Dimensions: 810x613
Number of bands: 1
Coordinate Reference System (CRS): None


  0%|          | 0/11930 [00:00<?, ?it/s]

100%|██████████| 11930/11930 [00:16<00:00, 738.52it/s] 


In [10]:
combined_assets['site_pga'] = combined_assets['site_pga'] /100 # convert from %g to g
print(combined_assets['site_pga'].describe())

count    11930.000000
mean         0.069018
std          0.111651
min          0.000000
25%          0.000000
50%          0.017882
75%          0.102377
max          0.653852
Name: site_pga, dtype: float64


In [11]:
output_dir = os.path.join(eq_scenario_dir, expansion_scenario)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_fp = os.path.join(output_dir, "combined_assets_with_pga.csv")
combined_assets.to_csv(output_fp, index=False)

In [12]:
output_dir

'/resnick/groups/enceladus/jyzhao/CAPowerResilience/Data/BSSC2014_scenarios/bssc2014ssanandreasnsbssbbgc_m7p64_se/conservative'